# Data-Driven Market Analyzer | Supervisor (Hierarchical)

In [1]:
import sys
import subprocess
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# Web search tool (free, no API key; requires `duckduckgo-search` package)
search_tool = DuckDuckGoSearchResults(name="web_search")

In [5]:
# Code execution tool (NOT sandboxed -- for demo purposes only; use Docker/E2B in production)
BLOCKED_PATTERNS = ["import os", "import sys", "import subprocess", "open(", "__import__"]

In [6]:
@tool
def run_code(code: str) -> str:
    """Execute Python code and return the output. Use for calculations, data analysis, and generating tables."""
    if any(pattern in code for pattern in BLOCKED_PATTERNS):
        return "Error: Potentially unsafe code detected. Only math and data operations are allowed."
    result = subprocess.run(
        [sys.executable, "-c", code],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode != 0:
        return f"Error:\n{result.stderr}"
    return result.stdout.strip() or "Code executed successfully (no output)."

In [7]:
# Note: `name` is required for multi-agent systems (supervisor/swarm identify agents by name).
# If `create_agent` does not accept `name`, use `create_react_agent` from `langgraph.prebuilt` instead.
research_agent = create_agent(
    model=model,
    tools=[search_tool],
    name="researcher",
    system_prompt=(
        "You are a market research specialist. Search the web for data on job markets, "
        "salaries, industry trends, and economic indicators. Return detailed findings with numbers."
    )
)

In [8]:
code_agent = create_agent(
    model=model,
    tools=[run_code],
    name="coder",
    system_prompt=(
        "You are a data analyst who writes Python code. When given research data, "
        "write code to calculate statistics, comparisons, percentage differences, "
        "and format results into clear tables. Always print the output."
    )
)

In [9]:
# Supervisor coordinates: researcher finds data, coder analyzes it, back-and-forth
workflow = create_supervisor(
    agents=[research_agent, code_agent],
    model=model,
    prompt=(
        "You are a market analysis project manager. Your goal is to produce a data-driven "
        "market comparison report. Workflow:\n"
        "1. Send the researcher to gather data on BOTH sides of the comparison\n"
        "2. Send the coder to calculate differences, ratios, and build comparison tables\n"
        "3. If the coder needs more data, send the researcher again\n"
        "4. Compile the final analysis with both qualitative insights and quantitative data"
    )
)

In [10]:
app = workflow.compile()

In [11]:
# Plot the workflow
plot_mermaid(app)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	coder(coder)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	coder --> supervisor;
	researcher --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> coder;
	supervisor -.-> researcher;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [12]:
result = app.invoke({
    "messages": [{
        "role": "user",
        "content": "Compare the tech job market in the US vs UK: average salaries for software engineers, demand trends, and top hiring cities"
    }]
})

In [13]:
print(result["messages"][-1].content)

### Market Comparison Report: Tech Job Market in the US vs UK

#### Average Salaries for Software Engineers
- **United States**: The average salary for a software engineer is approximately $130,000 annually, with variations based on skill levels and regions.
- **United Kingdom**: Specific average salary data is less clear, but major employers and locations tend to offer competitive remuneration.

#### Demand Trends
- **United States**: An upward trend in tech hiring with an 11% increase in job postings, highlighting a robust market, particularly in Texas.
- **United Kingdom**: Growth in tech jobs across multiple sectors, with increased female participation enhancing diversity in the workforce.

#### Top Hiring Cities
- **United States**: San Francisco, Seattle, New York City, Austin, and Boston are leading tech hubs known for their dynamic job markets.
- **United Kingdom**: London remains the primary tech hub, with emerging cities contributing to the landscape.

### Key Insights
1. **S

In [14]:
stream_invoke(app, {
    "messages": [{
        "role": "user",
        "content": "Compare the tech job market in the US vs UK: average salaries for software engineers, demand trends, and top hiring cities"
    }]
})

────────────────────────────────────────────────────────────────────────────────

  STREAMING EXECUTION

────────────────────────────────────────────────────────────────────────────────

┌─ HUMAN

│ 

...

└────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────

  EXECUTION COMPLETE

────────────────────────────────────────────────────────────────────────────────

{'messages': [HumanMessage(content='Compare the tech job market in the US vs UK: average salaries for software engineers, demand trends, and top hiring cities', additional_kwargs={}, response_metadata={}, id='6102a465-071e-443d-b028-1dd155fb26c3'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 166, 'total_tokens': 179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2ca5b70601', 'id': 'chatcmpl-DSU45gEGZ9edYVTppbHeUG44AFgg4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='supervisor', id='lc_run--019d6ed7-bad3-7e42-9ce5-3d7f32cdba03-0', tool_calls=[{'name': 'transfer_to_researcher', 'args': {}, 'id': 'call_st4BIpm7FHV